# LLM-Based Personalized Tutor for K-12 Short Answer Questions

### Models Tested:
- GPT-2 Small (Weak Baseline)
- Flan-T5 Small (Zero Shot)
- Flan-T5 Small (Fine Tuned)
- Flan-T5 Base (Zero Shot)
- mT5 Small (Zero Shot)

### Evaluation Metrics:
- ROUGE-1, ROUGE-2, ROUGE-L
- BLEU Score
- Human Evaluation (will do ourselves)
- Error Analysis


# SECTION 1: SETUP AND DATASET


In [1]:
# STEP 1.1: Check GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU RAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('WARNING: No GPU found!')
    print('Go to Runtime -> Change Runtime Type -> T4 GPU')

Device: cuda
GPU: Tesla T4
GPU RAM: 15.64 GB


In [2]:
# STEP 1.2: Install Libraries
!pip install transformers torch sentencepiece rouge-score evaluate scikit-learn nltk -q
print('All libraries installed!')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
All libraries installed!


In [3]:
# STEP 1.3: Import Libraries
import pandas as pd
import numpy as np
import torch
import gc
import nltk
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from evaluate import load
from sklearn.model_selection import train_test_split
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('All libraries imported!')

All libraries imported!


In [4]:
# STEP 1.4: Upload and Load Dataset
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('k12_dataset.csv')
print(f'Dataset loaded!')
print(f'Total samples:  {len(df)}')
print(f'Columns:        {list(df.columns)}')
print()
print('Subject distribution:')
print(df['subject'].value_counts())
print()
print('Grade distribution:')
print(df['grade_level'].value_counts())

Saving k12_dataset.csv to k12_dataset.csv
Dataset loaded!
Total samples:  100
Columns:        ['id', 'question', 'student_answer', 'correct_answer', 'grade_level', 'subject', 'misconception_type', 'expected_feedback']

Subject distribution:
subject
Math       52
Science    48
Name: count, dtype: int64

Grade distribution:
grade_level
Grade 4    28
Grade 5    25
Grade 6    23
Grade 7    12
Grade 3    11
Grade 8     1
Name: count, dtype: int64


In [5]:
# STEP 1.5: Train Test Split
# Zero Shot evaluation: 100 samples
# Fine Tuning training: 80 samples

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

zero_shot_df = df.head(100).reset_index(drop=True)

# Use first 80 samples for fine tuning training
finetune_train_df = train_df.head(80).reset_index(drop=True)

print(f'Total train samples:       {len(train_df)}')
print(f'Total test samples:        {len(test_df)}')
print(f'Zero shot eval samples:    {len(zero_shot_df)}')
print(f'Fine tuning train samples: {len(finetune_train_df)}')


Total train samples:       80
Total test samples:        20
Zero shot eval samples:    100
Fine tuning train samples: 80


In [6]:
# STEP 1.6: Define Prompt Templates
# PROMPT A - Basic Baseline
def prompt_A(row):
    return f"""explain why this answer is wrong and what the correct answer is:
question: {row['question']}
wrong answer: {row['student_answer']}
correct answer: {row['correct_answer']}"""

# PROMPT B - Role Based
def prompt_B(row):
    return f"""you are a {row['grade_level']} teacher.
explain to the student why their answer is wrong and guide them to the correct answer.
question: {row['question']}
student answer: {row['student_answer']}
correct answer: {row['correct_answer']}"""

# PROMPT C - Structured
def prompt_C(row):
    return f"""you are a {row['grade_level']} {row['subject']} teacher.
the student made a {row['misconception_type']}.
explain why the student answer is wrong and correct it simply.
question: {row['question']}
student answer: {row['student_answer']}
correct answer: {row['correct_answer']}"""

In [7]:
# STEP 1.7: Define Evaluation Functions
rouge = load('rouge')

def compute_rouge(predictions, references):
    results = rouge.compute(
        predictions=predictions,
        references=references
    )
    return {
        'ROUGE-1': round(results['rouge1'], 4),
        'ROUGE-2': round(results['rouge2'], 4),
        'ROUGE-L': round(results['rougeL'], 4)
    }

def compute_bleu(predictions, references):
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    smoothie = SmoothingFunction().method1
    refs  = [[ref.split()] for ref in references]
    hyps  = [pred.split() for pred in predictions]
    score = corpus_bleu(refs, hyps, smoothing_function=smoothie)
    return {'BLEU': round(score, 4)}

def evaluate_all(predictions, references):
    rouge_scores = compute_rouge(predictions, references)
    bleu_scores  = compute_bleu(predictions, references)
    return {**rouge_scores, **bleu_scores}

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()
    print('Memory cleared!')

print('Evaluation functions defined!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Evaluation functions defined!


In [8]:
# STEP 1.8: Initialize Results Storage
# This dictionary stores all results for final comparison
all_results = {}
all_predictions = {}
print('Results storage initialized!')

Results storage initialized!


---
# SECTION 2: GPT-2 SMALL (WEAK BASELINE)
---

In [9]:
# STEP 2.1: Load GPT-2 Small
clear_memory()
print('Loading GPT-2 Small...')

gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model     = GPT2LMHeadModel.from_pretrained('gpt2')
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model = gpt2_model.to(device)
gpt2_model.eval()

print(f'GPT-2 Small loaded!')
print(f'Parameters: {sum(p.numel() for p in gpt2_model.parameters())/1e6:.1f} Million')

Memory cleared!
Loading GPT-2 Small...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 Small loaded!
Parameters: 124.4 Million


In [10]:
# STEP 2.2: GPT-2 Prompt and Generation Function
def gpt2_prompt(row):
    return f"""Teacher feedback for a {row['grade_level']} student:
Question: {row['question']}
Student answered: {row['student_answer']}
Correct answer: {row['correct_answer']}
Teacher said:"""

def generate_gpt2(row, max_new_tokens=150):
    prompt = gpt2_prompt(row)
    inputs = gpt2_tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=512
    ).to(device)

    input_length = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = gpt2_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=False,
            no_repeat_ngram_size=3,
            repetition_penalty=1.5,
            length_penalty=2.0,
            pad_token_id=gpt2_tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][input_length:]
    return gpt2_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print('GPT-2 generation function defined!')

GPT-2 generation function defined!


In [11]:
# STEP 2.3: GPT-2 Test on 1 Sample
sample = df.iloc[0]
print('GPT-2 SMALL - 1 SAMPLE TEST')
print('=' * 60)
print(f'Question:          {sample["question"]}')
print(f'Student Answer:    {sample["student_answer"]}')
print(f'Correct Answer:    {sample["correct_answer"]}')
print(f'Expected Feedback: {sample["expected_feedback"]}')
print()
print(f'GPT-2 Output: {generate_gpt2(sample)}')
print('=' * 60)

GPT-2 SMALL - 1 SAMPLE TEST
Question:          What do plants need to make their own food?
Student Answer:    Plants only need water to make food.
Correct Answer:    Plants need sunlight, water, and carbon dioxide to make food.
Expected Feedback: Good try! You remembered water which is correct. However plants need three things not just one. They also need sunlight for energy and carbon dioxide from the air. Remember: sunlight + water + carbon dioxide = food for the plant!

GPT-2 Output: Plants don't need to be able to produce food in order to feed themselves. They need to have enough energy to make the food they need.
Answer: Plants can't produce food because they are not able to use their own energy. Plants need to eat what they need to survive and grow.
Students asked: What is the most important thing you want your students to know about plant life?
Teachers answered: The most important things are:
1. Plant life.
2. Food.
3. Water.
4. Carbon dioxide.
5. Insects.
6. Trees.
7. Birds.
8

In [12]:
# STEP 2.4: GPT-2 Run on Zero Shot Eval Set (100 Samples)
print('Running GPT-2 on zero shot eval set (100 samples)...')
gpt2_preds = []
references = zero_shot_df['expected_feedback'].tolist()

for i, row in zero_shot_df.iterrows():
    gpt2_preds.append(generate_gpt2(row))
    if (i + 1) % 10 == 0:
        print(f'Processed {i + 1}/{len(zero_shot_df)} samples...')

gpt2_scores = evaluate_all(gpt2_preds, references)
all_results['GPT-2 Small'] = gpt2_scores
all_predictions['GPT-2 Small'] = gpt2_preds

print()
print('GPT-2 Small Scores (100 samples):')
print(gpt2_scores)


Running GPT-2 on zero shot eval set (100 samples)...
Processed 10/100 samples...
Processed 20/100 samples...
Processed 30/100 samples...
Processed 40/100 samples...
Processed 50/100 samples...
Processed 60/100 samples...
Processed 70/100 samples...
Processed 80/100 samples...
Processed 90/100 samples...
Processed 100/100 samples...

GPT-2 Small Scores (100 samples):
{'ROUGE-1': np.float64(0.1657), 'ROUGE-2': np.float64(0.0256), 'ROUGE-L': np.float64(0.1076), 'BLEU': 0.0058}


In [13]:
# STEP 2.5: Clear GPT-2 from Memory
del gpt2_model
del gpt2_tokenizer
clear_memory()
print('GPT-2 removed from memory!')

Memory cleared!
GPT-2 removed from memory!


---
# SECTION 3: FLAN-T5 SMALL (ZERO SHOT)
---

In [43]:
# STEP 3.1: Load Flan-T5 Small
clear_memory()
print('Loading Flan-T5 Small...')

ft5s_tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-small')
ft5s_model     = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-small')
ft5s_model     = ft5s_model.to(device)
ft5s_model.eval()

print(f'Flan-T5 Small loaded!')
print(f'Parameters: {sum(p.numel() for p in ft5s_model.parameters())/1e6:.1f} Million')

Memory cleared!
Loading Flan-T5 Small...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Flan-T5 Small loaded!
Parameters: 77.0 Million


In [15]:
# STEP 3.2: Flan-T5 Small Generation Function
def generate_flanT5(row, tokenizer, model, prompt_fn, max_new_tokens=150):
    inputs = tokenizer(
        prompt_fn(row),
        return_tensors='pt',
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=False,
            no_repeat_ngram_size=3,
            repetition_penalty=1.5,
            length_penalty=2.0
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print('Flan-T5 generation function defined!')

Flan-T5 generation function defined!


In [16]:
# STEP 3.3: Flan-T5 Small Test on 1 Sample
sample = df.iloc[0]
print('FLAN-T5 SMALL ZERO SHOT - 1 SAMPLE TEST')
print('=' * 60)
print(f'Question:          {sample["question"]}')
print(f'Student Answer:    {sample["student_answer"]}')
print(f'Correct Answer:    {sample["correct_answer"]}')
print(f'Expected Feedback: {sample["expected_feedback"]}')
print()
print(f'Prompt A: {generate_flanT5(sample, ft5s_tokenizer, ft5s_model, prompt_A)}')
print(f'Prompt B: {generate_flanT5(sample, ft5s_tokenizer, ft5s_model, prompt_B)}')
print(f'Prompt C: {generate_flanT5(sample, ft5s_tokenizer, ft5s_model, prompt_C)}')
print('=' * 60)

FLAN-T5 SMALL ZERO SHOT - 1 SAMPLE TEST
Question:          What do plants need to make their own food?
Student Answer:    Plants only need water to make food.
Correct Answer:    Plants need sunlight, water, and carbon dioxide to make food.
Expected Feedback: Good try! You remembered water which is correct. However plants need three things not just one. They also need sunlight for energy and carbon dioxide from the air. Remember: sunlight + water + carbon dioxide = food for the plant!

Prompt A: carbon dioxide is a renewable source of energy for plants to grow.
Prompt B: carbon dioxide is a source of energy for plants to grow and thrive in the soils that surround them.doeing this will help plant populations become more sustainable, which can be done by cutting back on their own food sources such as fertilizer or other organic products like cornstarch (which has been proven over many years) rather than using pesticide-polluting chemicals used with crops; it'll also reduce greenhouse gas 

In [17]:
# STEP 3.4: Flan-T5 Small Run on Zero Shot Eval Set (100 Samples)
print('Running Flan-T5 Small Zero Shot on 100 samples...')

ft5s_preds_A, ft5s_preds_B, ft5s_preds_C = [], [], []

for i, row in zero_shot_df.iterrows():
    ft5s_preds_A.append(generate_flanT5(row, ft5s_tokenizer, ft5s_model, prompt_A))
    ft5s_preds_B.append(generate_flanT5(row, ft5s_tokenizer, ft5s_model, prompt_B))
    ft5s_preds_C.append(generate_flanT5(row, ft5s_tokenizer, ft5s_model, prompt_C))
    if (i + 1) % 10 == 0:
        print(f'Processed {i + 1}/{len(zero_shot_df)} samples...')

scores_ft5s_A = evaluate_all(ft5s_preds_A, references)
scores_ft5s_B = evaluate_all(ft5s_preds_B, references)
scores_ft5s_C = evaluate_all(ft5s_preds_C, references)

all_results['Flan-T5 Small ZS Prompt A'] = scores_ft5s_A
all_results['Flan-T5 Small ZS Prompt B'] = scores_ft5s_B
all_results['Flan-T5 Small ZS Prompt C'] = scores_ft5s_C
all_predictions['Flan-T5 Small ZS Prompt A'] = ft5s_preds_A
all_predictions['Flan-T5 Small ZS Prompt B'] = ft5s_preds_B
all_predictions['Flan-T5 Small ZS Prompt C'] = ft5s_preds_C

print()
print(f'Prompt A: {scores_ft5s_A}')
print(f'Prompt B: {scores_ft5s_B}')
print(f'Prompt C: {scores_ft5s_C}')


Running Flan-T5 Small Zero Shot on 100 samples...
Processed 10/100 samples...
Processed 20/100 samples...
Processed 30/100 samples...
Processed 40/100 samples...
Processed 50/100 samples...
Processed 60/100 samples...
Processed 70/100 samples...
Processed 80/100 samples...
Processed 90/100 samples...
Processed 100/100 samples...

Prompt A: {'ROUGE-1': np.float64(0.1724), 'ROUGE-2': np.float64(0.0426), 'ROUGE-L': np.float64(0.1142), 'BLEU': 0.0165}
Prompt B: {'ROUGE-1': np.float64(0.1538), 'ROUGE-2': np.float64(0.0366), 'ROUGE-L': np.float64(0.1017), 'BLEU': 0.0076}
Prompt C: {'ROUGE-1': np.float64(0.1855), 'ROUGE-2': np.float64(0.0504), 'ROUGE-L': np.float64(0.1285), 'BLEU': 0.0113}


---
# SECTION 4: FLAN-T5 SMALL (FINE TUNED)
---

In [44]:
# STEP 4.1: Prepare Dataset Class for Fine Tuning (20 Training Samples)
class FeedbackDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.data      = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        input_text  = prompt_C(row)
        target_text = str(row['expected_feedback'])

        input_enc = self.tokenizer(
            input_text,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        target_enc = self.tokenizer(
            target_text,
            max_length=150,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        labels = target_enc['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids':      input_enc['input_ids'].squeeze(),
            'attention_mask': input_enc['attention_mask'].squeeze(),
            'labels':         labels
        }

# Use only 80 samples for fine tuning
train_dataset = FeedbackDataset(finetune_train_df, ft5s_tokenizer)
train_loader  = DataLoader(train_dataset, batch_size=2, shuffle=True)
print(f'Fine tuning samples:  {len(finetune_train_df)}')
print(f'Train batches:        {len(train_loader)}')
print('Dataset ready for fine tuning!')


Fine tuning samples:  80
Train batches:        40
Dataset ready for fine tuning!


In [45]:
# STEP 4.2: Fine Tuning Loop
clear_memory()

EPOCHS        = 10
LEARNING_RATE = 5e-5

ft5s_model.train()
optimizer   = AdamW(ft5s_model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=5,
    num_training_steps=total_steps
)

print('Starting fine tuning...')
print(f'Model:         Flan-T5 Small')
print(f'Epochs:        {EPOCHS}')
print(f'Learning Rate: {LEARNING_RATE}')
print(f'Batch Size:    2')
print(f'Total Steps:   {total_steps}')
print()

train_losses = []

for epoch in range(EPOCHS):
    ft5s_model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = ft5s_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ft5s_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if batch_idx % 10 == 0:
            torch.cuda.empty_cache()

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}')

print()
print('Fine tuning completed!')

Memory cleared!
Starting fine tuning...
Model:         Flan-T5 Small
Epochs:        10
Learning Rate: 5e-05
Batch Size:    2
Total Steps:   400

Epoch 1/10 - Loss: 9.5083
Epoch 2/10 - Loss: 9.0843
Epoch 3/10 - Loss: 8.8687
Epoch 4/10 - Loss: 8.7357
Epoch 5/10 - Loss: 8.6351
Epoch 6/10 - Loss: 8.5503
Epoch 7/10 - Loss: 8.4869
Epoch 8/10 - Loss: 8.4458
Epoch 9/10 - Loss: 8.4224
Epoch 10/10 - Loss: 8.4062

Fine tuning completed!


In [20]:
# STEP 4.3: Save Fine Tuned Model
ft5s_model.save_pretrained('flan_t5_small_finetuned')
ft5s_tokenizer.save_pretrained('flan_t5_small_finetuned')
print('Fine tuned model saved!')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine tuned model saved!


In [21]:
# STEP 4.4: Test Fine Tuned Model on 1 Sample
ft5s_model.eval()
sample = df.iloc[0]

print('FLAN-T5 SMALL FINE TUNED - 1 SAMPLE TEST')
print('=' * 60)
print(f'Question:          {sample["question"]}')
print(f'Student Answer:    {sample["student_answer"]}')
print(f'Correct Answer:    {sample["correct_answer"]}')
print(f'Expected Feedback: {sample["expected_feedback"]}')
print()
print(f'Fine Tuned Output: {generate_flanT5(sample, ft5s_tokenizer, ft5s_model, prompt_C)}')
print('=' * 60)

FLAN-T5 SMALL FINE TUNED - 1 SAMPLE TEST
Question:          What do plants need to make their own food?
Student Answer:    Plants only need water to make food.
Correct Answer:    Plants need sunlight, water, and carbon dioxide to make food.
Expected Feedback: Good try! You remembered water which is correct. However plants need three things not just one. They also need sunlight for energy and carbon dioxide from the air. Remember: sunlight + water + carbon dioxide = food for the plant!

Fine Tuned Output: It is a way to make food. They are not in the water and it can be so that they will have more energy for them! So you do just as we get there, which means one or both of yours has no air on its body but only because all plants need an area at their place by this time with some light from each side (no matter what). You don't see how many things go up when two people find out about different ones: like rain-water; also use oxygen before other animals may take off over again? The plant d

In [22]:
# STEP 4.5: Fine Tuned Evaluation on 20 Test Samples
# Fine tuning was done on 80 samples so we evaluate on first 20 test samples
ft5s_ft_eval_df = test_df.head(20).reset_index(drop=True)
ft5s_ft_refs    = ft5s_ft_eval_df['expected_feedback'].tolist()

print('Running Fine Tuned Flan-T5 Small on 20 test samples...')
ft5s_ft_preds = []

for i, row in ft5s_ft_eval_df.iterrows():
    ft5s_ft_preds.append(generate_flanT5(row, ft5s_tokenizer, ft5s_model, prompt_C))
    if (i + 1) % 5 == 0:
        print(f'Processed {i + 1}/{len(ft5s_ft_eval_df)} samples...')

scores_ft5s_ft = evaluate_all(ft5s_ft_preds, ft5s_ft_refs)
all_results['Flan-T5 Small Fine Tuned'] = scores_ft5s_ft
all_predictions['Flan-T5 Small Fine Tuned'] = ft5s_ft_preds

print()
print(f'Fine Tuned Scores (20 samples): {scores_ft5s_ft}')


Running Fine Tuned Flan-T5 Small on 20 test samples...
Processed 5/20 samples...
Processed 10/20 samples...
Processed 15/20 samples...
Processed 20/20 samples...

Fine Tuned Scores (20 samples): {'ROUGE-1': np.float64(0.24), 'ROUGE-2': np.float64(0.0325), 'ROUGE-L': np.float64(0.1251), 'BLEU': 0.0047}


In [23]:
# STEP 4.6: Clear Flan-T5 Small from Memory
del ft5s_model
del ft5s_tokenizer
clear_memory()
print('Flan-T5 Small removed from memory!')

Memory cleared!
Flan-T5 Small removed from memory!


---
# SECTION 5: FLAN-T5 BASE (ZERO SHOT)
---

In [24]:
# STEP 5.1: Load Flan-T5 Base
clear_memory()
print('Loading Flan-T5 Base...')

ft5b_tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
ft5b_model     = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')
ft5b_model     = ft5b_model.to(device)
ft5b_model.eval()

print(f'Flan-T5 Base loaded!')
print(f'Parameters: {sum(p.numel() for p in ft5b_model.parameters())/1e6:.1f} Million')

Memory cleared!
Loading Flan-T5 Base...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Flan-T5 Base loaded!
Parameters: 247.6 Million


In [25]:
# STEP 5.2: Flan-T5 Base Test on 1 Sample
sample = df.iloc[0]
print('FLAN-T5 BASE ZERO SHOT - 1 SAMPLE TEST')
print('=' * 60)
print(f'Question:          {sample["question"]}')
print(f'Student Answer:    {sample["student_answer"]}')
print(f'Correct Answer:    {sample["correct_answer"]}')
print(f'Expected Feedback: {sample["expected_feedback"]}')
print()
print(f'Prompt A: {generate_flanT5(sample, ft5b_tokenizer, ft5b_model, prompt_A)}')
print(f'Prompt B: {generate_flanT5(sample, ft5b_tokenizer, ft5b_model, prompt_B)}')
print(f'Prompt C: {generate_flanT5(sample, ft5b_tokenizer, ft5b_model, prompt_C)}')
print('=' * 60)

FLAN-T5 BASE ZERO SHOT - 1 SAMPLE TEST
Question:          What do plants need to make their own food?
Student Answer:    Plants only need water to make food.
Correct Answer:    Plants need sunlight, water, and carbon dioxide to make food.
Expected Feedback: Good try! You remembered water which is correct. However plants need three things not just one. They also need sunlight for energy and carbon dioxide from the air. Remember: sunlight + water + carbon dioxide = food for the plant!

Prompt A: Sunlight and carbon dioxide are essential for plants to make their own food. So the final answer may vary based on what you're looking at, but in most cases it does not have an impact upon your overall health or well-balanced eating habits (ie low blood pressure), weight gain/loss of body mass index(BMI) as measured by how much water they drink each day during daylight hours (9:15 am–17:00 pm ET). If there is any doubt about this information please let us know so that we can update our research a

In [26]:
# STEP 5.3: Flan-T5 Base Run on Zero Shot Eval Set (100 Samples)
print('Running Flan-T5 Base Zero Shot on 100 samples...')

ft5b_preds_A, ft5b_preds_B, ft5b_preds_C = [], [], []

for i, row in zero_shot_df.iterrows():
    ft5b_preds_A.append(generate_flanT5(row, ft5b_tokenizer, ft5b_model, prompt_A))
    ft5b_preds_B.append(generate_flanT5(row, ft5b_tokenizer, ft5b_model, prompt_B))
    ft5b_preds_C.append(generate_flanT5(row, ft5b_tokenizer, ft5b_model, prompt_C))
    if (i + 1) % 10 == 0:
        print(f'Processed {i + 1}/{len(zero_shot_df)} samples...')

scores_ft5b_A = evaluate_all(ft5b_preds_A, references)
scores_ft5b_B = evaluate_all(ft5b_preds_B, references)
scores_ft5b_C = evaluate_all(ft5b_preds_C, references)

all_results['Flan-T5 Base ZS Prompt A'] = scores_ft5b_A
all_results['Flan-T5 Base ZS Prompt B'] = scores_ft5b_B
all_results['Flan-T5 Base ZS Prompt C'] = scores_ft5b_C
all_predictions['Flan-T5 Base ZS Prompt A'] = ft5b_preds_A
all_predictions['Flan-T5 Base ZS Prompt B'] = ft5b_preds_B
all_predictions['Flan-T5 Base ZS Prompt C'] = ft5b_preds_C

print()
print(f'Prompt A: {scores_ft5b_A}')
print(f'Prompt B: {scores_ft5b_B}')
print(f'Prompt C: {scores_ft5b_C}')


Running Flan-T5 Base Zero Shot on 100 samples...
Processed 10/100 samples...
Processed 20/100 samples...
Processed 30/100 samples...
Processed 40/100 samples...
Processed 50/100 samples...
Processed 60/100 samples...
Processed 70/100 samples...
Processed 80/100 samples...
Processed 90/100 samples...
Processed 100/100 samples...

Prompt A: {'ROUGE-1': np.float64(0.2162), 'ROUGE-2': np.float64(0.0491), 'ROUGE-L': np.float64(0.1361), 'BLEU': 0.0181}
Prompt B: {'ROUGE-1': np.float64(0.2246), 'ROUGE-2': np.float64(0.0461), 'ROUGE-L': np.float64(0.1383), 'BLEU': 0.0189}
Prompt C: {'ROUGE-1': np.float64(0.2292), 'ROUGE-2': np.float64(0.0515), 'ROUGE-L': np.float64(0.1374), 'BLEU': 0.0183}


In [27]:
# STEP 5.4: Clear Flan-T5 Base from Memory
del ft5b_model
del ft5b_tokenizer
clear_memory()
print('Flan-T5 Base removed from memory!')

Memory cleared!
Flan-T5 Base removed from memory!


---
# SECTION 6: mT5 SMALL (ZERO SHOT)
---

In [28]:
# STEP 6.1: Load mT5 Small
clear_memory()
print('Loading mT5 Small...')

mt5_tokenizer = AutoTokenizer.from_pretrained('google/mt5-small')
mt5_model     = AutoModelForSeq2SeqLM.from_pretrained('google/mt5-small')
mt5_model     = mt5_model.to(device)
mt5_model.eval()

print(f'mT5 Small loaded!')
print(f'Parameters: {sum(p.numel() for p in mt5_model.parameters())/1e6:.1f} Million')

Memory cleared!
Loading mT5 Small...


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

mT5 Small loaded!
Parameters: 556.3 Million


In [29]:
# STEP 6.2: mT5 Specific Prompts
# mT5 works better with lowercase key value style prompts
def mt5_prompt_A(row):
    return f"""question: {row['question']}
student answer: {row['student_answer']}
correct answer: {row['correct_answer']}
provide teacher feedback:"""

def mt5_prompt_B(row):
    return f"""grade: {row['grade_level']}
question: {row['question']}
student answer: {row['student_answer']}
correct answer: {row['correct_answer']}
provide encouraging teacher feedback:"""

def mt5_prompt_C(row):
    return f"""grade: {row['grade_level']}
subject: {row['subject']}
misconception: {row['misconception_type']}
question: {row['question']}
student answer: {row['student_answer']}
correct answer: {row['correct_answer']}
provide detailed encouraging teacher feedback:"""

print('mT5 prompts defined!')

mT5 prompts defined!


In [30]:
# STEP 6.3: mT5 Test on 1 Sample
sample = test_df.iloc[0]
print('mT5 SMALL ZERO SHOT - 1 SAMPLE TEST')
print('=' * 60)
print(f'Question:          {sample["question"]}')
print(f'Student Answer:    {sample["student_answer"]}')
print(f'Correct Answer:    {sample["correct_answer"]}')
print(f'Expected Feedback: {sample["expected_feedback"]}')
print()
print(f'Prompt A: {generate_flanT5(sample, mt5_tokenizer, mt5_model, mt5_prompt_A)}')
print(f'Prompt B: {generate_flanT5(sample, mt5_tokenizer, mt5_model, mt5_prompt_B)}')
print(f'Prompt C: {generate_flanT5(sample, mt5_tokenizer, mt5_model, mt5_prompt_C)}')
print('=' * 60)

mT5 SMALL ZERO SHOT - 1 SAMPLE TEST
Question:          Solve: x + 9 = 20.
Student Answer:    x = 29.
Correct Answer:    x = 11.
Expected Feedback: Almost there! To find x subtract 9 from both sides: x = 20 - 9 = 11. It looks like you added 9 to 20 instead of subtracting. Remember: to cancel addition on one side subtract from both sides!

Prompt A: <extra_id_0> question: Solve: x + 9 =
Prompt B: <extra_id_0> question: Solve: x + 9 =
Prompt C: <extra_id_0> question: Solve: x + 9 =


In [31]:
# STEP 6.4: mT5 Run on Zero Shot Eval Set (100 Samples)
print('Running mT5 Small Zero Shot on 100 samples...')

mt5_preds_A, mt5_preds_B, mt5_preds_C = [], [], []

for i, row in zero_shot_df.iterrows():
    mt5_preds_A.append(generate_flanT5(row, mt5_tokenizer, mt5_model, mt5_prompt_A))
    mt5_preds_B.append(generate_flanT5(row, mt5_tokenizer, mt5_model, mt5_prompt_B))
    mt5_preds_C.append(generate_flanT5(row, mt5_tokenizer, mt5_model, mt5_prompt_C))
    if (i + 1) % 10 == 0:
        print(f'Processed {i + 1}/{len(zero_shot_df)} samples...')

scores_mt5_A = evaluate_all(mt5_preds_A, references)
scores_mt5_B = evaluate_all(mt5_preds_B, references)
scores_mt5_C = evaluate_all(mt5_preds_C, references)

all_results['mT5 Small ZS Prompt A'] = scores_mt5_A
all_results['mT5 Small ZS Prompt B'] = scores_mt5_B
all_results['mT5 Small ZS Prompt C'] = scores_mt5_C
all_predictions['mT5 Small ZS Prompt A'] = mt5_preds_A
all_predictions['mT5 Small ZS Prompt B'] = mt5_preds_B
all_predictions['mT5 Small ZS Prompt C'] = mt5_preds_C

print()
print(f'Prompt A: {scores_mt5_A}')
print(f'Prompt B: {scores_mt5_B}')
print(f'Prompt C: {scores_mt5_C}')


Running mT5 Small Zero Shot on 100 samples...
Processed 10/100 samples...
Processed 20/100 samples...
Processed 30/100 samples...
Processed 40/100 samples...
Processed 50/100 samples...
Processed 60/100 samples...
Processed 70/100 samples...
Processed 80/100 samples...
Processed 90/100 samples...
Processed 100/100 samples...

Prompt A: {'ROUGE-1': np.float64(0.0726), 'ROUGE-2': np.float64(0.0287), 'ROUGE-L': np.float64(0.0652), 'BLEU': 0.0}
Prompt B: {'ROUGE-1': np.float64(0.0799), 'ROUGE-2': np.float64(0.0266), 'ROUGE-L': np.float64(0.0704), 'BLEU': 0.0}
Prompt C: {'ROUGE-1': np.float64(0.0639), 'ROUGE-2': np.float64(0.0173), 'ROUGE-L': np.float64(0.0565), 'BLEU': 0.0}


In [32]:
# STEP 6.5: Clear mT5 from Memory
del mt5_model
del mt5_tokenizer
clear_memory()
print('mT5 Small removed from memory!')

Memory cleared!
mT5 Small removed from memory!


---
# SECTION 7: FINAL RESULTS COMPARISON
---

In [33]:
# STEP 7.1: Complete Results Table
results_table = pd.DataFrame(all_results).T
results_table = results_table.reset_index()
results_table.columns = ['Model', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BLEU']

print('=' * 75)
print('           COMPLETE RESULTS TABLE FOR PAPER')
print('=' * 75)
print(results_table.to_string(index=False))
print('=' * 75)
print()
print('Best ROUGE-1 Model:', results_table.loc[results_table['ROUGE-1'].idxmax(), 'Model'])
print('Best ROUGE-2 Model:', results_table.loc[results_table['ROUGE-2'].idxmax(), 'Model'])
print('Best ROUGE-L Model:', results_table.loc[results_table['ROUGE-L'].idxmax(), 'Model'])
print('Best BLEU Model:   ', results_table.loc[results_table['BLEU'].idxmax(), 'Model'])

           COMPLETE RESULTS TABLE FOR PAPER
                    Model  ROUGE-1  ROUGE-2  ROUGE-L   BLEU
              GPT-2 Small   0.1657   0.0256   0.1076 0.0058
Flan-T5 Small ZS Prompt A   0.1724   0.0426   0.1142 0.0165
Flan-T5 Small ZS Prompt B   0.1538   0.0366   0.1017 0.0076
Flan-T5 Small ZS Prompt C   0.1855   0.0504   0.1285 0.0113
 Flan-T5 Small Fine Tuned   0.2400   0.0325   0.1251 0.0047
 Flan-T5 Base ZS Prompt A   0.2162   0.0491   0.1361 0.0181
 Flan-T5 Base ZS Prompt B   0.2246   0.0461   0.1383 0.0189
 Flan-T5 Base ZS Prompt C   0.2292   0.0515   0.1374 0.0183
    mT5 Small ZS Prompt A   0.0726   0.0287   0.0652 0.0000
    mT5 Small ZS Prompt B   0.0799   0.0266   0.0704 0.0000
    mT5 Small ZS Prompt C   0.0639   0.0173   0.0565 0.0000

Best ROUGE-1 Model: Flan-T5 Small Fine Tuned
Best ROUGE-2 Model: Flan-T5 Base ZS Prompt C
Best ROUGE-L Model: Flan-T5 Base ZS Prompt B
Best BLEU Model:    Flan-T5 Base ZS Prompt B


In [34]:
# STEP 7.2: Training Loss Table
losses_df = pd.DataFrame({
    'Epoch': list(range(1, EPOCHS+1)),
    'Training Loss': train_losses
})
print('TRAINING LOSS PER EPOCH')
print('=' * 30)
print(losses_df.to_string(index=False))
print('=' * 30)

TRAINING LOSS PER EPOCH
 Epoch  Training Loss
     1       9.089516
     2       8.393110
     3       8.140134


---
# SECTION 8: ERROR ANALYSIS
---

In [35]:
# STEP 8.1: Compute All ROUGE Metrics Per Sample
# We analyze Fine Tuned Flan-T5 Small outputs as it is our best model
from evaluate import load as load_metric
rouge_metric = load_metric('rouge')

print('ERROR ANALYSIS - FINE TUNED FLAN-T5 SMALL')
print('=' * 60)
print()

per_sample_rouge1 = []
per_sample_rouge2 = []
per_sample_rougeL = []

for pred, ref in zip(ft5s_ft_preds, ft5s_ft_refs):
    score = rouge_metric.compute(predictions=[pred], references=[ref])
    per_sample_rouge1.append(score['rouge1'])
    per_sample_rouge2.append(score['rouge2'])
    per_sample_rougeL.append(score['rougeL'])

test_df_copy = ft5s_ft_eval_df.copy()
test_df_copy['generated_feedback'] = ft5s_ft_preds
test_df_copy['rouge1_score']       = per_sample_rouge1
test_df_copy['rouge2_score']       = per_sample_rouge2
test_df_copy['rougeL_score']       = per_sample_rougeL

print('Per-sample ROUGE metrics computed!')
print(f'Average ROUGE-1: {sum(per_sample_rouge1)/len(per_sample_rouge1):.4f}')
print(f'Average ROUGE-2: {sum(per_sample_rouge2)/len(per_sample_rouge2):.4f}')
print(f'Average ROUGE-L: {sum(per_sample_rougeL)/len(per_sample_rougeL):.4f}')


ERROR ANALYSIS - FINE TUNED FLAN-T5 SMALL

Per-sample ROUGE metrics computed!
Average ROUGE-1: 0.2394
Average ROUGE-2: 0.0326
Average ROUGE-L: 0.1257


In [36]:
# STEP 8.2: Best and Worst Outputs (by ROUGE-L)
best_samples  = test_df_copy.nlargest(3, 'rougeL_score')
worst_samples = test_df_copy.nsmallest(3, 'rougeL_score')

print('TOP 3 BEST OUTPUTS (Highest ROUGE-L)')
print('=' * 65)
for _, row in best_samples.iterrows():
    print(f'ROUGE-1: {row["rouge1_score"]:.4f}  ROUGE-2: {row["rouge2_score"]:.4f}  ROUGE-L: {row["rougeL_score"]:.4f}')
    print(f'Question:          {row["question"]}')
    print(f'Student Answer:    {row["student_answer"]}')
    print(f'Expected Feedback: {row["expected_feedback"]}')
    print(f'Generated:         {row["generated_feedback"]}')
    print()

print()
print('TOP 3 WORST OUTPUTS (Lowest ROUGE-L)')
print('=' * 65)
for _, row in worst_samples.iterrows():
    print(f'ROUGE-1: {row["rouge1_score"]:.4f}  ROUGE-2: {row["rouge2_score"]:.4f}  ROUGE-L: {row["rougeL_score"]:.4f}')
    print(f'Question:          {row["question"]}')
    print(f'Student Answer:    {row["student_answer"]}')
    print(f'Expected Feedback: {row["expected_feedback"]}')
    print(f'Generated:         {row["generated_feedback"]}')
    print()


TOP 3 BEST OUTPUTS (Highest ROUGE-L)
ROUGE-1: 0.3380  ROUGE-2: 0.0580  ROUGE-L: 0.1972
Question:          What is the value of the digit 2 in 8241?
Student Answer:    The value is 200.
Expected Feedback: Excellent work! You got it right. In 8241 the digit 2 is in the hundreds place and its value is 2 × 100 = 200. Remember: always identify the place position first then multiply to find the value!
Generated:         It is a number. The value of the 1 and 2 are in it! You can have one or two to find that for all at 3: 5 + 4 = 7- 8 * 10= 6 (2+4)

ROUGE-1: 0.3243  ROUGE-2: 0.0556  ROUGE-L: 0.1892
Question:          What is the value of the digit 4 in 4823?
Student Answer:    The value is 4.
Expected Feedback: Almost there! The digit 4 is in the thousands place so its value is 4 × 1000 = 4000 not just 4. The value of a digit depends on its position. Remember: always multiply the digit by its place value to find its actual value!
Generated:         It is a number. The value of the 1 and 2 in 

In [37]:
# STEP 8.3: Analysis by Subject
print('ALL ROUGE SCORES BY SUBJECT')
print('=' * 50)
subject_scores = test_df_copy.groupby('subject')[[
    'rouge1_score', 'rouge2_score', 'rougeL_score'
]].mean().round(4)
subject_scores.columns = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
print(subject_scores.to_string())

print()
print('ALL ROUGE SCORES BY GRADE LEVEL')
print('=' * 50)
for grade_group, grades in [
    ('Grade 3-5', ['Grade 3', 'Grade 4', 'Grade 5']),
    ('Grade 6-8', ['Grade 6', 'Grade 7', 'Grade 8'])
]:
    idx = test_df_copy['grade_level'].isin(grades)
    if idx.sum() > 0:
        r1 = test_df_copy[idx]['rouge1_score'].mean()
        r2 = test_df_copy[idx]['rouge2_score'].mean()
        rl = test_df_copy[idx]['rougeL_score'].mean()
        print(f'{grade_group:12s}  ROUGE-1: {r1:.4f}  ROUGE-2: {r2:.4f}  ROUGE-L: {rl:.4f}')

print()
print('ALL ROUGE SCORES BY MISCONCEPTION TYPE')
print('=' * 50)
misconception_scores = test_df_copy.groupby('misconception_type')[[
    'rouge1_score', 'rouge2_score', 'rougeL_score'
]].mean().round(4)
misconception_scores.columns = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
print(misconception_scores.sort_values('ROUGE-L', ascending=False).to_string())


ALL ROUGE SCORES BY SUBJECT
         ROUGE-1  ROUGE-2  ROUGE-L
subject                           
Math      0.2625   0.0369   0.1443
Science   0.2240   0.0297   0.1133

ALL ROUGE SCORES BY GRADE LEVEL
Grade 3-5     ROUGE-1: 0.2481  ROUGE-2: 0.0346  ROUGE-L: 0.1273
Grade 6-8     ROUGE-1: 0.2045  ROUGE-2: 0.0246  ROUGE-L: 0.1196

ALL ROUGE SCORES BY MISCONCEPTION TYPE
                                         ROUGE-1  ROUGE-2  ROUGE-L
misconception_type                                                
Gravity and force confusion               0.2299   0.0706   0.1839
Place value misunderstanding              0.2788   0.0378   0.1578
Geometry angle confusion                  0.2920   0.0444   0.1460
Multiplication and division confusion     0.3299   0.0421   0.1443
Basic algebra error                       0.1693   0.0164   0.1349
Confusion between body organs/functions   0.2014   0.0359   0.1311
Water cycle misunderstanding              0.2261   0.0000   0.1217
Fraction addition error     

In [41]:
# STEP 8.4: Automated Error Categorization
print('ERROR CATEGORY DISTRIBUTION')
print('=' * 50)

for model_name, preds in all_predictions.items():
    print(f'\nError Analysis: {model_name}')
    print('=' * 50)

    # ← Move this INSIDE the loop
    error_categories = {
        'Incomplete output (< 5 words)':   0,
        'Sentinel tokens (extra_id)':      0,
        'Very short output (5-10 words)':  0,
        'Normal output (> 10 words)':      0,
    }

    for pred in preds:
        word_count = len(pred.split())
        if '<extra_id_0>' in pred:
            error_categories['Sentinel tokens (extra_id)'] += 1
        elif word_count < 5:
            error_categories['Incomplete output (< 5 words)'] += 1
        elif word_count <= 10:
            error_categories['Very short output (5-10 words)'] += 1
        else:
            error_categories['Normal output (> 10 words)'] += 1

    for category, count in error_categories.items():
        pct = count / len(preds) * 100
        print(f'  {category:40s}: {count:3d}  ({pct:.1f}%)')
print()
print('NOTE: Hallucination, Copying student answer, and Wrong correction')
print('      require manual inspection. See generated outputs above.')


ERROR CATEGORY DISTRIBUTION

Error Analysis: GPT-2 Small
  Incomplete output (< 5 words)           :   0  (0.0%)
  Sentinel tokens (extra_id)              :   0  (0.0%)
  Very short output (5-10 words)          :   0  (0.0%)
  Normal output (> 10 words)              : 100  (100.0%)

Error Analysis: Flan-T5 Small ZS Prompt A
  Incomplete output (< 5 words)           :  30  (30.0%)
  Sentinel tokens (extra_id)              :   0  (0.0%)
  Very short output (5-10 words)          :  17  (17.0%)
  Normal output (> 10 words)              :  53  (53.0%)

Error Analysis: Flan-T5 Small ZS Prompt B
  Incomplete output (< 5 words)           :  36  (36.0%)
  Sentinel tokens (extra_id)              :   0  (0.0%)
  Very short output (5-10 words)          :  18  (18.0%)
  Normal output (> 10 words)              :  46  (46.0%)

Error Analysis: Flan-T5 Small ZS Prompt C
  Incomplete output (< 5 words)           :  21  (21.0%)
  Sentinel tokens (extra_id)              :   0  (0.0%)
  Very short output (

---
# SECTION 9: SAVE ALL RESULTS AND DOWNLOAD
---

In [39]:
# STEP 9.1: Save All Outputs to CSV
final_output_df = zero_shot_df.copy()
final_output_df['GPT2_output']              = all_predictions['GPT-2 Small']
final_output_df['FlanT5_Small_ZS_PromptA']  = all_predictions['Flan-T5 Small ZS Prompt A']
final_output_df['FlanT5_Small_ZS_PromptB']  = all_predictions['Flan-T5 Small ZS Prompt B']
final_output_df['FlanT5_Small_ZS_PromptC']  = all_predictions['Flan-T5 Small ZS Prompt C']
final_output_df['FlanT5_Base_ZS_PromptA']   = all_predictions['Flan-T5 Base ZS Prompt A']
final_output_df['FlanT5_Base_ZS_PromptB']   = all_predictions['Flan-T5 Base ZS Prompt B']
final_output_df['FlanT5_Base_ZS_PromptC']   = all_predictions['Flan-T5 Base ZS Prompt C']
final_output_df['mT5_Small_ZS_PromptA']     = all_predictions['mT5 Small ZS Prompt A']
final_output_df['mT5_Small_ZS_PromptB']     = all_predictions['mT5 Small ZS Prompt B']
final_output_df['mT5_Small_ZS_PromptC']     = all_predictions['mT5 Small ZS Prompt C']
final_output_df.to_csv('k12_zero_shot_outputs.csv', index=False)

# Save fine tuned outputs separately (20 samples)
ft_output_df = ft5s_ft_eval_df.copy()
ft_output_df['FlanT5_Small_FineTuned'] = all_predictions['Flan-T5 Small Fine Tuned']
ft_output_df.to_csv('k12_finetuned_outputs.csv', index=False)

# Save ROUGE and BLEU scores
results_table.to_csv('final_rouge_bleu_scores.csv', index=False)

# Save training losses
losses_df.to_csv('training_losses.csv', index=False)

# Save error analysis
test_df_copy.to_csv('error_analysis.csv', index=False)

print('All files saved!')
print()
print('Files created:')
print('  1. k12_zero_shot_outputs.csv      - Zero shot outputs (100 samples)')
print('  2. k12_finetuned_outputs.csv       - Fine tuned outputs (20 samples)')
print('  3. final_rouge_bleu_scores.csv     - ROUGE and BLEU scores table')
print('  4. training_losses.csv             - Fine tuning loss per epoch')
print('  5. error_analysis.csv              - Error analysis with all ROUGE scores')


All files saved!

Files created:
  1. k12_zero_shot_outputs.csv      - Zero shot outputs (100 samples)
  2. k12_finetuned_outputs.csv       - Fine tuned outputs (20 samples)
  3. final_rouge_bleu_scores.csv     - ROUGE and BLEU scores table
  4. training_losses.csv             - Fine tuning loss per epoch
  5. error_analysis.csv              - Error analysis with all ROUGE scores


In [40]:
# STEP 9.2: Download All Files
from google.colab import files

files.download('k12_zero_shot_outputs.csv')
files.download('k12_finetuned_outputs.csv')
files.download('final_rouge_bleu_scores.csv')
files.download('training_losses.csv')
files.download('error_analysis.csv')

print()
print('All files downloaded!')
print()
print('Files for your paper:')
print('  1. k12_zero_shot_outputs.csv      - All zero shot model outputs (100 samples)')
print('  2. k12_finetuned_outputs.csv       - Fine tuned model outputs (20 samples)')
print('  3. final_rouge_bleu_scores.csv     - ROUGE and BLEU scores for all models')
print('  4. training_losses.csv             - Training loss per epoch')
print('  5. error_analysis.csv              - Error analysis with ROUGE-1/2/L per sample')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All files downloaded!

Files for your paper:
  1. k12_zero_shot_outputs.csv      - All zero shot model outputs (100 samples)
  2. k12_finetuned_outputs.csv       - Fine tuned model outputs (20 samples)
  3. final_rouge_bleu_scores.csv     - ROUGE and BLEU scores for all models
  4. training_losses.csv             - Training loss per epoch
  5. error_analysis.csv              - Error analysis with ROUGE-1/2/L per sample
